Example of how to parse the files and load data in dataframes.

In [44]:
import os


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('TkAgg')
%matplotlib notebook


In [45]:
data_path = os.getcwd() + os.sep + 'data'
print(data_path)

D:\Projekt-Projekt\Examensarbete\data


In [46]:
# Loading data functions
def loader(data_path, blacklist):
    data = []
    all_data_csv = pd.read_csv(data_path + os.sep + 'metadata_tracks.csv')
    for name in os.listdir(data_path):
        if name != '.DS_Store' and 'csv' not in name and 'txt' not in name:
            print('---------------------------------------------')
            print("Loading walking data for subject: ",name)
            new_path = data_path + os.sep + name
            for name_in in os.listdir(new_path):
                if name_in != '.DS_Store' and name_in not in blacklist:
                    print("Loading test: ", name_in)
                    motion_csv = pd.read_csv(new_path + os.sep + name_in + os.sep + 'motion.csv')
                    position_csv = pd.read_csv(new_path + os.sep + name_in + os.sep + 'positions.csv')
                    orientation_csv = pd.read_csv(new_path + os.sep + name_in + os.sep + 'orientation.csv')
                    steps_csv = pd.read_csv(new_path + os.sep + name_in + os.sep + 'steps.csv')
                    event_csv = pd.read_csv(new_path + os.sep + name_in + os.sep + 'events.csv')

                    anon_test = walking_test(
                        name_in, event_csv,
                        motion_csv,
                        orientation_csv,
                        position_csv,
                        steps_csv
                    )
                    data.append(anon_test)

    return data, all_data_csv

class walking_test:
    def __init__(self, filename, events, motion, orientation, positions, steps): #
        self.filename = filename
        self.events = events
        self.motion = motion
        self.orientation = orientation
        self.positions = positions
        self.steps = steps

In [47]:
# Uploading metadata and determining blacklist
metadata = pd.read_csv(data_path + os.sep + 'metadata_tracks.csv')
print("Metadata columns: ", metadata.columns)
print(metadata.head())
noIMU = metadata[metadata['hasMotion'] == False]['testName'].tolist()
noGPS = metadata[metadata['hasGNSS'] == False]['testName'].tolist()
blacklist = noIMU + noGPS
print("Blacklist: ", blacklist)

Metadata columns:  Index(['subject', 'testID', 'testName', 'isPatient', 'distanceReference',
       'hasMotion', 'hasGNSS', 'device', 'distanceByApp', 'totSteps',
       'path curvature', 'total_gaps_time_inertial', 'total_gaps_time_gnss',
       'gt_type', 'country', 'gnss_anonimized', 'duration [s]', 'fs_acc',
       'fs_gnss', 'fs_steps', 'average_walking_speed', 'smartphone_position',
       'smartphone app'],
      dtype='object')
   subject  testID testName  isPatient  distanceReference  hasMotion  hasGNSS  \
0        0       0      0_0       True                623      False     True   
1        0       1      0_1       True                598      False     True   
2        0       2      0_2       True                584      False     True   
3        0       3      0_3       True                577      False     True   
4        0       4      0_4       True                590      False     True   

             device  distanceByApp  totSteps  ...  gt_type  country  \
0 

In [48]:
# Loading data
selection, info_all_tests = loader(data_path, blacklist=blacklist)
print("Successfully loaded data")

---------------------------------------------
Loading walking data for subject:  subject_0
---------------------------------------------
Loading walking data for subject:  subject_1
---------------------------------------------
Loading walking data for subject:  subject_10
Loading test:  10_0
Loading test:  10_1
Loading test:  10_10
Loading test:  10_11
Loading test:  10_12
Loading test:  10_13
Loading test:  10_14
Loading test:  10_15
Loading test:  10_16
Loading test:  10_17
Loading test:  10_2
Loading test:  10_3
Loading test:  10_4
Loading test:  10_5
Loading test:  10_6
Loading test:  10_7
Loading test:  10_8
Loading test:  10_9
---------------------------------------------
Loading walking data for subject:  subject_11
Loading test:  11_0
Loading test:  11_1
Loading test:  11_10
Loading test:  11_11
Loading test:  11_12
Loading test:  11_13
Loading test:  11_14
Loading test:  11_15
Loading test:  11_2
Loading test:  11_3
Loading test:  11_4
Loading test:  11_5
Loading test:  11_6


In [73]:
from scipy.interpolate import interp1d
import geopy

# Example of what a walking test contains:
print("Selection[0] contains: ")
print("Filename: ", selection[0].filename)
print("Events: ", selection[0].events.head())
print("Motion: ", selection[0].motion.head())
print("Orientation: ", selection[0].orientation.head())
print("Positions: ", selection[0].positions.head())
print("Steps: ", selection[0].steps.head())

time_stamps = selection[0].motion['ms'].values
gps_time_stamps = selection[0].positions['ms'].values


def calculate_XYZ(motion):
    xyz = list(zip(motion['accelX'], motion['accelY'], motion['accelZ']))
    xyz_array = np.array(xyz)
    magnitude_value = np.linalg.norm(xyz_array, axis = 1)

    averaged_magnitudes = []
    for i in range(0, len(magnitude_value), 2):
        avg = np.mean(magnitude_value[i:i+2])
        averaged_magnitudes.append(avg)

    return np.array(averaged_magnitudes)

from geopy.distance import geodesic
from filterpy.kalman import KalmanFilter

def calculate_distance(position):
    coords = list(zip(position['latitude'], position['longitude']))
    distances = []

    for i in range(len(coords) - 1):
        dist = geopy.distance.geodesic(coords[i], coords[i+1]).meters
        distances.append(dist)

    return np.array(distances)

def interpolate_distance(distance_traveled, motion_time_stamps, gps_time_stamps):
    # Check lengths
    print(f"Length of distance_traveled: {len(distance_traveled)}")
    print(f"Length of gps_time_stamps: {len(gps_time_stamps)}")

    if len(distance_traveled) != len(gps_time_stamps):
        print("Mismatch in lengths, aligning arrays...")

        # Trim the longer array (gps_time_stamps)
        gps_time_stamps = gps_time_stamps[:-1]

        print(f"After trimming, Length of distance_traveled: {len(distance_traveled)}")
        print(f"After trimming, Length of gps_time_stamps: {len(gps_time_stamps)}")

    # Now interpolate gps_time_stamps to match motion_time_stamps
    interp_func = interp1d(gps_time_stamps, distance_traveled, kind='linear', fill_value="extrapolate")
    interpolated_distances = interp_func(motion_time_stamps)

    return interpolated_distances

distance_traveled = calculate_distance(selection[0].positions)

def setup_KalmanFiltering(dt,accel_magnitude):

    kf = KalmanFilter(dim_x=3,dim_z=2) #initialization for the kalman object x = position, velocity acceleration, z = 2 is the values we want to update in our case position and acceleration

    kf.F = np.array([[1,dt, 0.5 * dt**2], #shows the evolution of the variables, position, velocity and acceleration
                    [0,1,dt],
                    [0,0,1]])

    kf.P = np.array([[100,0,0],
                     [0,100,0],
                     [0,0,100]]) #kalmans confidence of estimated initial state

    kf.H = np.array([[1, 0, 0],
                     [0, 0, 1]])  # Distance, speed, acceleration, means that we observe only distance and acceleration

    kf.R = np.array([[100,0],
                     [0,100]]) # amount of noise reduction applied

    kf.B = np.array([[0],[0],[1]]) # the control matrix, 0,0,1 means we modify the influence of acceleration

    accel_Q = np.std(accel_magnitude) #represents how noisy the acceleration measure are
    dt2 = dt**2
    dt3 = dt2 * dt / 2
    dt4 = dt2 * dt2 / 4

    kf.Q = np.array([[dt4 * accel_Q, dt3 * accel_Q, dt2 * accel_Q / 2], #this models the uncertainty in the systems dynamics
                     [dt3 * accel_Q, dt2 * accel_Q, dt * accel_Q],
                     [dt2 * accel_Q / 2, dt * accel_Q, accel_Q]])

    initial_position = 0
    initial_velocity = 0
    initial_acceleration = accel_magnitude[0]
    kf.x = np.array([initial_position, initial_velocity, initial_acceleration])

    return kf

def update_Kalman_with_position(kf, accel_magnitude,distance_traveled ,time_stamps,gps_time_stamps):
    total_distance = 0.0
    previous_position = kf.x[0]
    total_loops = 0
    interpolated_distances = interpolate_distance(distance_traveled, time_stamps,gps_time_stamps)

    for i in range(1, len(accel_magnitude)):
        dt = (time_stamps[i] - time_stamps[i-1]) / 1000.0

        accel = accel_magnitude[i]
        distance_step = interpolated_distances[i]
        kf.F = np.array([[1, dt, 0.5 * dt**2],
                         [0, 1, dt],
                         [0, 0, 1]])

        sigma_acc = np.std(accel_magnitude)
        dt2 = dt**2
        dt3 = dt2 * dt / 2
        dt4 = dt2 * dt2 / 4

        kf.Q = np.array([
            [dt4 * sigma_acc, dt3 * sigma_acc, dt2 * sigma_acc / 2],
            [dt3 * sigma_acc, dt2 * sigma_acc, dt * sigma_acc],
            [dt2 * sigma_acc / 2, dt * sigma_acc, sigma_acc]
        ])
        total_loops +=1

        kf.predict()

        z = np.array([distance_step, accel])
        kf.update(z)

        current_position = kf.x[0]

        distance_traveled = abs(current_position - previous_position)
        total_distance += distance_traveled

        previous_position = current_position

    return kf, total_distance,total_loops

dt = 1.0
accel_magnitude = calculate_XYZ(selection[0].motion)
position = selection[0].positions
kf = setup_KalmanFiltering(dt, accel_magnitude)
kf,total_distance,total_loops = update_Kalman_with_position(kf, accel_magnitude, distance_traveled, time_stamps,gps_time_stamps)

print("total loops", total_loops)
print("Total Distance Traveled (after Kalman filter):", total_distance)

Selection[0] contains: 
Filename:  10_0
Events:     Unnamed: 0  signalStart  testStart        testEnd
0           0            0     1481.0  361167.000055
Motion:     Unnamed: 0     ms  accelX  accelY  accelZ  accelWithGX  accelWithGY  \
0           0  178.0    -0.1     0.0     0.2         -0.2          3.2   
1           1  192.0    -0.1     0.0     0.3         -0.2          3.2   
2           2  200.0    -0.1    -0.1     0.3         -0.3          3.1   
3           3  218.0    -0.1    -0.2     0.4         -0.2          2.9   
4           4  234.0    -0.1    -0.2     0.4         -0.3          2.9   

   accelWithGZ  rotRateAlpha  rotRateBeta  rotRateGamma  interval  
0          9.4          -2.6          0.8           3.3        16  
1          9.4          -2.6          0.8           3.3        16  
2          9.6          -2.8         -1.4           3.4        16  
3          9.7          -1.6         -2.1           2.4        16  
4          9.8           0.5         -3.4          

In [ ]:
# Example of how to plot position data
fig = plt.figure()
ax = fig.add_subplot(121)
ax.plot(selection[0].positions['longitude'], selection[0].positions['latitude'],'o--')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('GNSS')

ax = fig.add_subplot(122)
ax.plot(selection[0].motion['accelZ'],'o--', label='Z')
ax.plot(selection[0].motion['accelX'],'o--', label='X')
ax.plot(selection[0].motion['accelY'],'o--', label='Y')
ax.set_xlabel('Time')
ax.set_ylabel('Acceleration')
ax.legend()
ax.set_title('Motion')
plt.show()


In [ ]:
metadata.head()

In [ ]:
metadata['smartphone_position'].value_counts()